<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/GPT_repl_Karpathy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-01-25 22:35:21--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2026-01-25 22:35:21 (170 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [6]:
import torch.nn as nn
from torch.nn import functional as F

In [7]:
# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
# ------------


In [8]:

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)


# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]


In [9]:

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [10]:

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [11]:

model = BigramLanguageModel(vocab_size)
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


In [20]:

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % (eval_interval) == 299:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 299: train loss 2.4592, val loss 2.4890
step 599: train loss 2.4567, val loss 2.4962
step 899: train loss 2.4553, val loss 2.4785
step 1199: train loss 2.4546, val loss 2.4841
step 1499: train loss 2.4549, val loss 2.4926
step 1799: train loss 2.4564, val loss 2.4925
step 2099: train loss 2.4514, val loss 2.4850
step 2399: train loss 2.4611, val loss 2.4853
step 2699: train loss 2.4571, val loss 2.4938
step 2999: train loss 2.4568, val loss 2.4828


ME my, lindwo.
YO:
TAst f acodit l, theighaig trvaimeloreatang.

LINA:
CENGANI m ch noree wind RI I htow'd th BELatorth:
Norire my ak beror mo m.
MELird.
Whanessloul fis beemp d t me we weirg onecldil ivanfut;
D:
Whiethier,

And me y s roun kered wncende ifome thid bickit s is,
ALLUSobuthofateso wapowht wod ser alegunderm aiknd! cithe.
And tixe gun inde no

Le: tomepllavey t.
ASouy corimangich stourith CH:
YONGor! therthad by crinos y, thaththes byou my y, wo htherde blliothes I: weayondil ispt


In [208]:
B,T,C = 4,8,2
tril=torch.tril(torch.ones(T,T))
tril=tril.masked_fill(tril==0,float("-inf"))
F.softmax(tril, dim=1)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [211]:
wei=torch.tril(torch.ones(T,T)).T
wei=(wei/torch.sum(wei,0)).T
wei=wei.masked_fill(tril==0,float("-inf"))
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [175]:
z

tensor([[[ 0.8184,  1.0985],
         [-0.2854, -0.1761],
         [ 0.0594, -0.4640],
         [ 1.8395, -1.0400],
         [-0.1174, -1.1583],
         [ 1.9659,  2.6100],
         [-0.3750,  0.8334],
         [-0.4985, -0.7871]],

        [[-0.5900,  1.7716],
         [-1.2227, -1.6319],
         [ 2.0053,  0.1763],
         [-0.4853, -2.3136],
         [-0.4704, -0.6100],
         [ 0.5775, -0.5709],
         [-1.9073, -0.1508],
         [ 0.2109,  0.0715]],

        [[-0.2540, -1.2298],
         [-2.1148, -1.2404],
         [-0.9541, -1.1091],
         [ 0.7916,  1.0120],
         [ 0.5176, -1.2566],
         [ 1.2696,  0.1482],
         [-0.6790, -0.2681],
         [ 0.3335,  0.0977]],

        [[-1.8052,  1.6689],
         [ 1.9041, -0.0574],
         [ 0.1940, -0.5093],
         [ 0.7194,  0.7063],
         [-0.6852,  0.0188],
         [-0.7107, -0.2750],
         [-0.5075, -0.2151],
         [-0.0076,  0.2549]]])

In [190]:
wei

tensor([[1.0000, 0.5000, 0.3333, 0.2500, 0.2000, 0.1667, 0.1429, 0.1250],
        [0.0000, 0.5000, 0.3333, 0.2500, 0.2000, 0.1667, 0.1429, 0.1250],
        [0.0000, 0.0000, 0.3333, 0.2500, 0.2000, 0.1667, 0.1429, 0.1250],
        [0.0000, 0.0000, 0.0000, 0.2500, 0.2000, 0.1667, 0.1429, 0.1250],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.2000, 0.1667, 0.1429, 0.1250],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1667, 0.1429, 0.1250],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1429, 0.1250],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1250]])

In [136]:
z.view(4,8,2)@d

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x2 and 8x8)